## Summary: Bronze Ingestion Step
Purpose
This script ensures:
- Consistent naming conventions
- Preservation of both full and summary datasets
- Decompression of .gz files for downstream readability



What It Does
- Resolves Project Paths
- Uses pathlib to dynamically locate the project root and create a data/bronze folder.
- Ensures portability across machines and avoids hardcoded paths.
- Defines Snapshot Source
- Targets a specific Airbnb data snapshot (2025-03-04) for reproducibility.
- Separates data files (e.g. listings, calendar, reviews) from visualisation files (e.g. neighbourhoods, summaries).
- Maps File URLs to Local Names
- Creates a dictionary of filenames and their corresponding download URLs.
- Applies consistent naming to avoid overwriting or ambiguity (e.g. listings_full.csv.gz to listings_full.csv).
- Downloads Each File
- Streams each file in chunks to avoid memory overload.
- Saves files with appropriate extensions (.csv, .csv.gz, .geojson).
- Decompresses .gz Files
- Automatically decompresses .csv.gz files to .csv using gzip and shutil.
- Keeps both compressed and decompressed versions for traceability.


Failsafes Built In
- Status Code Check: Skips any file that fails to download (status_code != 200) and prints a clear error message.
- Extension Logic: Dynamically assigns the correct file extension based on URL suffix.
- Directory Creation: Ensures bronze_dir exists before writing files (mkdir(parents=True, exist_ok=True)).
- Chunked Streaming: Prevents memory overload during large file downloads.
- Dual File Preservation: Keeps both .gz and .csv versions for audit and flexibility.


Alignment with Scalable, Reproducible Pipeline

- Modularity - Each file is handled independently; logic is reusable across datasets
- Reproducibility - Snapshot date is fixed; file naming is deterministic
- Auditability - Raw files are preserved alongside decompressed versions
- Portability - Uses pathlib to avoid OS-specific path issues
- Scalability - Easily extendable to new datasets or snapshots by updating the files dictionary
- Fail-Safe Execution - Gracefully handles download failures without breaking the loop

In [ ]:
# ---------------------------------------------------------
# BRONZE INGESTION STEP (RAW FILES, KEEP FULL + SUMMARY)
# Purpose: Download Airbnb datasets into Bronze with clear
# naming to avoid overwriting:
#   - listings_full.csv.gz → listings_full.csv
#   - reviews_full.csv.gz → reviews_full.csv
#   - calendar.csv.gz → calendar.csv
#   - listings_summary.csv
#   - reviews_summary.csv
#   - neighbourhoods.csv
#   - neighbourhoods.geojson
# ---------------------------------------------------------

import requests, gzip, shutil
from pathlib import Path

# Resolve project root Gold/Silver/Bronze
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
bronze_dir = project_root / "data" / "bronze"
bronze_dir.mkdir(parents=True, exist_ok=True)

snapshot_date = "2025-03-04"
base_data_url = f"http://data.insideairbnb.com/united-kingdom/england/london/{snapshot_date}/data"
base_vis_url  = f"http://data.insideairbnb.com/united-kingdom/england/london/{snapshot_date}/visualisations"

files = {
    "listings_full": f"{base_data_url}/listings.csv.gz",
    "calendar": f"{base_data_url}/calendar.csv.gz",
    "reviews_full": f"{base_data_url}/reviews.csv.gz",
    "listings_summary": f"{base_vis_url}/listings.csv",
    "reviews_summary": f"{base_vis_url}/reviews.csv",
    "neighbourhoods": f"{base_vis_url}/neighbourhoods.csv",
    "neighbourhoods_geo": f"{base_vis_url}/neighbourhoods.geojson"
}

for name, url in files.items():
    # Decide file extension
    if url.endswith(".gz"):
        raw_path = bronze_dir / f"{name}.csv.gz"
    elif url.endswith(".geojson"):
        raw_path = bronze_dir / f"{name}.geojson"
    else:
        raw_path = bronze_dir / f"{name}.csv"

    # Download
    r = requests.get(url, stream=True)
    if r.status_code != 200:
        print(f"❌ Failed to fetch {url} (status {r.status_code})")
        continue

    with open(raw_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

    # If compressed, also decompress to .csv
    if raw_path.suffix == ".gz":
        csv_path = raw_path.with_suffix("")  # removes .gz
        with gzip.open(raw_path, "rb") as f_in, open(csv_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        print(f"✅ Saved {name} → {raw_path} (compressed) and {csv_path} (decompressed)")
    else:
        print(f"✅ Saved {name} → {raw_path}")

✅ Saved listings_full → C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\bronze\listings_full.csv.gz (compressed) and C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\bronze\listings_full.csv (decompressed)
✅ Saved calendar → C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\bronze\calendar.csv.gz (compressed) and C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\bronze\calendar.csv (decompressed)
✅ Saved reviews_full → C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\bronze\reviews_full.csv.gz (compressed) and C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\bronze\reviews_full.csv (decompressed)
✅ Saved listings_summary → C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\bronze\listings_summary.csv
✅ Saved reviews_summary → C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\bronze\reviews_summary.csv
✅ Saved neighbourhoods → C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\bronze\neighbourhoods.csv
✅